### Step 1: Import Libraries

We import essential libraries:
- `tensorflow` and `keras` for model creation and training.
- `mobilenet_v3` for loading the MobileNetV3Small architecture.
- `EarlyStopping` and `ReduceLROnPlateau` are callbacks to prevent overfitting and adjust learning rate dynamically.


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

# Image shape
input_shape = (256, 256, 3)

# Create the base model
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=input_shape,
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # Freeze for now

# Build the full model
inputs = tf.keras.Input(shape=input_shape)
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(6, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


C:\Users\saima\anaconda3\envs\tfenv\lib\site-packages\keras\src\applications\mobilenet_v3.py:452: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ MobileNetV3Small (Functional)        │ (None, 8, 8, 576)           │         939,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 576)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 576)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 6)                   │           3,462 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 942,582 (3.60 MB)

 Trainable params: 3,462 (13.52 KB)

 Non-trainable params: 939,120 (3.58 MB)

### Step 2: Load Dataset

We use TensorFlow’s `image_dataset_from_directory` to load our trash classification dataset.
- We split the dataset into training (80%) and validation (20%) sets.
- Images are resized to 256x256.


In [6]:
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
     r"C:\Users\saima\OneDrive\Desktop\Garbage-Classification\Week 1\TrashType_Image_Dataset",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int",
    seed=42
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
     r"C:\Users\saima\OneDrive\Desktop\Garbage-Classification\Week 1\TrashType_Image_Dataset",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int",
    seed=42
)

# ✅ Extract class names BEFORE cache/shuffle/prefetch
class_names = raw_train_ds.class_names
print("Class names:", class_names)

# Then apply cache/shuffle
AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = raw_val_ds.cache().prefetch(buffer_size=AUTOTUNE)


Found 2527 files belonging to 6 classes.
Found 2527 files belonging to 6 classes.
Class names: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


### Step 3: Preprocess Data

We:
- Extract class names from training data.
- Cache and prefetch the datasets to improve performance.
- Shuffle the training dataset to ensure randomization.


In [7]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])


### Step 4: Load Pretrained MobileNetV3Small

We load the MobileNetV3Small model:
- `include_top=False` excludes the final classification layer.
- We freeze the base model so only custom layers are trainable initially.

### Step 5: Add Custom Classification Layers

We add:
- GlobalAveragePooling to reduce spatial dimensions.
- Dropout to prevent overfitting.
- Dense layer with softmax to classify into 6 trash categories.



In [8]:
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras import layers, models

# Load MobileNetV3Small (already frozen)
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(256, 256, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

# Build the full model
inputs = tf.keras.Input(shape=(256, 256, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ MobileNetV3Small (Functional)        │ (None, 8, 8, 576)           │         939,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 576)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 576)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │           3,462 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 942,582 (3.60 MB)

 Trainable params: 3,462 (13.52 KB)

 Non-trainable params: 939,120 (3.58 MB)

### Step 6: Compile and Train the Model

We compile the model with:
- Adam optimizer.
- Sparse categorical crossentropy (for integer labels).
- Accuracy metric.
Training uses early stopping and learning rate reduction callbacks.


In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(factor=0.5, patience=2)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 43s 429ms/step - accuracy: 0.3515 - loss: 1.6876 - val_accuracy: 0.7008 - val_loss: 0.8496 - learning_rate: 0.0010
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 30s 377ms/step - accuracy: 0.6694 - loss: 0.8990 - val_accuracy: 0.7582 - val_loss: 0.6777 - learning_rate: 0.0010
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 30s 381ms/step - accuracy: 0.7096 - loss: 0.7615 - val_accuracy: 0.7926 - val_loss: 0.5831 - learning_rate: 0.0010
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 30s 378ms/step - accuracy: 0.7702 - loss: 0.6554 - val_accuracy: 0.8176 - val_loss: 0.5274 - learning_rate: 0.0010
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 30s 379ms/step - accuracy: 0.7797 - loss: 0.6401 - val_accuracy: 0.8231 - val_loss: 0.5002 - learning_rate: 0.0010
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 30s 377ms/step - accuracy: 0.7711 - loss: 0.6092 - val_accuracy: 0.8362 - val_loss: 0.4774 - learning_rate: 0.0010
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 30s 381ms/step - accuracy: 0.8005 - loss: 0.

### Step 7: Fine-Tuning

After initial training, we unfreeze the base model to fine-tune it.
We use a lower learning rate (1e-5) to avoid overwriting pretrained weights.


In [10]:
# Unfreeze the base model for fine-tuning
base_model.trainable = True

# Recompile with lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - accuracy: 0.5622 - loss: 1.2698 - val_accuracy: 0.8662 - val_loss: 0.3950 - learning_rate: 1.0000e-05
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.6481 - loss: 0.9359 - val_accuracy: 0.8579 - val_loss: 0.4226 - learning_rate: 1.0000e-05
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.6837 - loss: 0.8339 - val_accuracy: 0.8472 - val_loss: 0.4529 - learning_rate: 1.0000e-05
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.7416 - loss: 0.7138 - val_accuracy: 0.8401 - val_loss: 0.4565 - learning_rate: 5.0000e-06


In [11]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"\n✅ Final Validation Accuracy: {val_acc * 100:.2f}%")


79/79 ━━━━━━━━━━━━━━━━━━━━ 14s 171ms/step - accuracy: 0.8737 - loss: 0.3801

✅ Final Validation Accuracy: 86.62%


### 🔁 Test Time Augmentation (TTA)
To improve prediction robustness, we applied TTA during inference by averaging predictions over several augmented versions of the same input image. This helps account for variations in orientation, lighting, etc.


In [12]:
import tensorflow as tf
import numpy as np

def tta_predict(model, image, num_augments=5):
    predictions = []
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (256, 256))

    for _ in range(num_augments):
        aug_img = tf.image.random_flip_left_right(image)
        aug_img = tf.image.random_brightness(aug_img, 0.1)
        aug_img = tf.image.random_contrast(aug_img, 0.9, 1.1)
        aug_img = tf.expand_dims(aug_img, axis=0)
        aug_img = tf.keras.applications.mobilenet_v3.preprocess_input(aug_img)
        predictions.append(model.predict(aug_img, verbose=0))

    return np.mean(predictions, axis=0)


In [15]:
correct = 0
total = 0

# Limit to first few batches only
for image_batch, label_batch in val_ds.take(5):  # take 5 batches (adjust if needed)
    for i in range(len(image_batch)):
        pred = tta_predict(model, image_batch[i])
        if np.argmax(pred) == label_batch[i].numpy():
            correct += 1
        total += 1

print(f"🔎 Approx. TTA Accuracy on Subset: {correct / total * 100:.2f}%")


🔎 Approx. TTA Accuracy on Subset: 90.00%


In [17]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


In [18]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"\n✅ Final Validation Accuracy: {val_acc * 100:.2f}%")


79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 184ms/step - accuracy: 0.8737 - loss: 0.3801

✅ Final Validation Accuracy: 86.62%


### Step 8: Save the Trained Model

We save the model in Keras' recommended format (`.keras`).
This model can now be loaded into a Streamlit app or used for inference.


In [23]:
model.save("smartbin_model.keras")  # Saves entire model (architecture + weights + optimizer)


### 📦 Model Conversion to TensorFlow Lite
We converted the trained Keras model to TFLite format to make it lightweight and optimized for deployment on edge devices or mobile platforms.


In [24]:
# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model
with open("smartbin_model.tflite", "wb") as f:
    f.write(tflite_model)


INFO:tensorflow:Assets written to: C:\Users\saima\AppData\Local\Temp\tmp4l4xfoam\assets


INFO:tensorflow:Assets written to: C:\Users\saima\AppData\Local\Temp\tmp4l4xfoam\assets


Saved artifact at 'C:\Users\saima\AppData\Local\Temp\tmp4l4xfoam'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='keras_tensor_355')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1582247748960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247747376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247750544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247746144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247748784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247755472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247659472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247662112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247658240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1582247660704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1